# Demo live — TimesFM-3 di fronte a uno shock

**Offline by design.** Questo notebook legge solo `results/*.parquet` — niente rete,
niente Colab, niente Hugging Face il giorno del talk. Se questa cella sopra fallisce,
il fix è rilanciare gli esperimenti (`scripts/02..05`), non aggiustare questo notebook.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from tfm3lab import config

shock = pd.read_parquet(config.RESULTS_DIR / "exp_shock_raw_predictions.parquet")
lag = pd.read_parquet(config.RESULTS_DIR / "exp_shock_adaptation_lag.parquet")
shock = shock[(shock["series"] == "SP500") & (shock["mode"] == "timesfm3_multivariate")]
print(sorted(shock["event"].unique()))

## Passo 1 — un evento pre-cutoff

Cambia `EVENT_PRE` qui sotto per scegliere un evento diverso dal vivo durante il talk.

In [ ]:
def plot_reaction(df: pd.DataFrame, event_name: str, ax_pair):
    sub = df[df["event"] == event_name].sort_values("offset")
    ax0, ax1 = ax_pair
    ax0.plot(sub["offset"], sub["actual"], color="#111827", linewidth=2, label="reale")
    ax0.plot(sub["offset"], sub["forecast"], color="#2563eb", marker="o", markersize=3, label="TimesFM-3")
    ax0.fill_between(sub["offset"], sub["q10"], sub["q90"], color="#2563eb", alpha=0.15, label="P10-P90")
    ax0.axvline(0, color="red", linestyle="--", label="evento")
    ax0.set_title(event_name)
    ax0.legend(fontsize=8)
    ax1.plot(sub["offset"], sub["abs_pct_error"], marker="o", color="#dc2626")
    ax1.axvline(0, color="black", linestyle="--")
    ax1.set_xlabel("offset (giorni dall'evento)")
    ax1.set_ylabel("errore % assoluto")


EVENT_PRE = "Crollo Covid"
fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
plot_reaction(shock, EVENT_PRE, axes)
plt.show()

## Passo 2 — lo stesso protocollo, un evento post-cutoff

Stesso codice, altro evento. Il confronto visivo è il punto della demo.

In [ ]:
EVENT_POST = "Shock dazi"
fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
plot_reaction(shock, EVENT_POST, axes)
plt.show()

## Passo 3 — affiancati, e l'adaptation lag

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 6), sharex="col")
plot_reaction(shock, EVENT_PRE, axes[:, 0])
plot_reaction(shock, EVENT_POST, axes[:, 1])
plt.tight_layout()
plt.show()

print("Adaptation lag medio per braccio (multiplier=1.5):")
print(lag[lag["multiplier"] == 1.5].groupby("arm")["adaptation_lag_days"].mean().to_string())

## Messaggio di chiusura

> I foundation model per serie temporali non sono oracoli. Sono sistemi di aggiornamento
> probabilistico: riconoscono pattern osservati, ma uno shock veramente nuovo diventa
> prevedibile solo dopo che ha iniziato a lasciare una traccia nei dati — e prima di
> crederci, bisogna verificare che non lo stia semplicemente ricordando.